# 04 - Recent reviews table

- First look at `STEAM_DB.STEAM_RAW.REVIEW_RECENT`.
- It comes from the same Steam endpoint as `review_summary`. This time I ask only for the reviews of the **last 30 days**, split into positive and negative.
- One row is one game in one run: `recent_positive`, `recent_negative` and `day_range` (30). The total is not stored. dbt adds positive + negative later.
- Steam does not send a count directly. For each game I send two requests (`review_type` 1 = positive, 2 = negative) and keep `total_matching` from the answer. All languages are counted.
- `collected_at` is one time for the whole run, like in `review_summary`.

- The data is one load from 2026-11-02 (load id `1793621677.9089065`), 81 Paradox games.
- The notebook only runs SELECT and DESCRIBE, so it changes nothing in Snowflake.

- Limits:
  - There is only one snapshot, so no trend yet.
  - The 30 days slide forward every day, so two days in a row share 29 days.
  - A game with few reviews gets a jumpy percentage.
  - Steam says it leaves out review bombs by default. I did not test how much that changes the counts.
  - Valve does not document this endpoint.

## 0. Connect to Snowflake

Same connection as notebooks 01 to 03. It reads the settings from `.env`, logs in as `STEAM_LOADER` and prints no secrets. Times are in UTC (Stockholm is UTC+1 now). Pick the project's `.venv` as the kernel.

In [1]:
import os
from pathlib import Path

import pandas as pd
import snowflake.connector

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [2]:
def find_repo_root() -> Path:
    """Walk up from the notebook's folder until we find pyproject.toml."""
    here = Path.cwd().resolve()
    return next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())


def load_env_file(path: Path) -> None:
    """Copy KEY=value lines from a .env file into os.environ (never prints values)."""
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


load_env_file(find_repo_root() / ".env")

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_LOADER_USER"],
    private_key_file=os.environ["SNOWFLAKE_LOADER_PRIVATE_KEY_PATH"],
    role=os.environ["SNOWFLAKE_LOADER_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="STEAM_RAW",
    session_parameters={"TIMEZONE": "UTC"},
)


def query(sql: str) -> pd.DataFrame:
    """Run one read-only statement and return the result as a DataFrame."""
    with conn.cursor() as cursor:
        cursor.execute(sql)
        columns = [column[0] for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)

In [3]:
# Sanity check: we should be the loader role, in STEAM_DB.STEAM_RAW.
query("""
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema
""")

,ROLE,WAREHOUSE,DATABASE,SCHEMA
0,STEAM_LOADER,STEAM_WH,STEAM_DB,STEAM_RAW


In [4]:
# The newest load. The queries below use it, so they keep working when more runs arrive.
LOAD_ID = query("SELECT MAX(_dlt_load_id) AS load_id FROM review_recent")["LOAD_ID"][0]
LOAD_ID

'1793621677.9089065'

## 1. Rows per load

- Every run adds one row per game. All rows of one run share one `collected_at`.

**💡Insights:**
- One load, 81 rows, 81 different games.
- All rows have the same `collected_at` (12:14:39 UTC), and `day_range` is 30 everywhere.
- The log of the run said the same: 81 fetched, 0 skipped, 0 failed.

In [5]:
query("""
SELECT _dlt_load_id, COUNT(*) AS rows_loaded, COUNT(DISTINCT steam_appid) AS distinct_games,
       MIN(collected_at) AS first_row, MAX(collected_at) AS last_row,
       MIN(day_range) AS min_day_range, MAX(day_range) AS max_day_range
FROM review_recent
GROUP BY _dlt_load_id
ORDER BY _dlt_load_id
""")

,_DLT_LOAD_ID,ROWS_LOADED,DISTINCT_GAMES,FIRST_ROW,LAST_ROW,MIN_DAY_RANGE,MAX_DAY_RANGE
0,1793621677.9089065,81,81,2026-11-02 12:14:39.403937+00:00,2026-11-02 12:14:39.403937+00:00,30,30


## 2. Duplicates

A game should show up once per run. This lists any game that shows up twice in the same load.

**💡Insights:**
- The result is empty, so there are no duplicates.

In [6]:
query("""
SELECT steam_appid, _dlt_load_id, COUNT(*) AS copies
FROM review_recent
GROUP BY steam_appid, _dlt_load_id
HAVING COUNT(*) > 1
""")

,STEAM_APPID,_DLT_LOAD_ID,COPIES


## 3. Column types

**💡Insights:**
- `COLLECTED_AT` is `TIMESTAMP_LTZ(9)`, the same as in the other tables.
- `STEAM_APPID`, `DAY_RANGE`, `RECENT_POSITIVE` and `RECENT_NEGATIVE` are `NUMBER(19,0)`.
- dlt added `_DLT_LOAD_ID` and `_DLT_ID`. Every other column can be NULL.

In [7]:
query("DESCRIBE TABLE review_recent")[["name", "type", "null?"]]

,name,type,null?
0,COLLECTED_AT,TIMESTAMP_LTZ(9),Y
1,STEAM_APPID,"NUMBER(19,0)",Y
2,DAY_RANGE,"NUMBER(19,0)",Y
3,RECENT_POSITIVE,"NUMBER(19,0)",Y
4,RECENT_NEGATIVE,"NUMBER(19,0)",Y
5,_DLT_LOAD_ID,VARCHAR(16777216),N
6,_DLT_ID,VARCHAR(16777216),N


## 4. Games with no recent reviews

- A 0 here is a real answer from Steam (`total_matching` was 0). It is not a missing value.

**💡Insights:**

| Game | `coming_soon` | Release date | Recent | All-time reviews |
|---|:---:|---|---:|---:|
| Afterworld | True | To be announced | 0 | 0 |
| Guardians of the Wild Sky | True | Coming soon | 0 | 0 |
| LEGO Skylines | True | Coming soon | 0 | 0 |
| Prison Architect 2 | True | To be announced | 0 | 0 |
| Knights of Pen and Paper 2: Free Edition | False | 21 Feb, 2018 | 0 | 0 |

- 5 games have 0 recent reviews: the 4 unreleased games and the Free Edition, which already had no reviews in notebook 02. All five also have 0 reviews of all time.
- The other 76 games have at least 1 review in the last 30 days (section 6).
- For unreleased games Steam also says 0. So "no reviews" and "no data" look the same here. To know if a game is unreleased, use `coming_soon` from `games`, like in notebook 03.

In [8]:
query(f"""
SELECT r.steam_appid, g.name, g.coming_soon, g.release_date_text,
       r.recent_positive, r.recent_negative, s.total_reviews AS all_time_reviews
FROM review_recent r
JOIN games g ON g.steam_appid = r.steam_appid AND g._dlt_load_id = r._dlt_load_id
JOIN review_summary s ON s.steam_appid = r.steam_appid AND s._dlt_load_id = r._dlt_load_id
WHERE r._dlt_load_id = '{LOAD_ID}'
  AND r.recent_positive + r.recent_negative = 0
ORDER BY g.coming_soon DESC, g.name
""")

,STEAM_APPID,NAME,COMING_SOON,RELEASE_DATE_TEXT,RECENT_POSITIVE,RECENT_NEGATIVE,ALL_TIME_REVIEWS
0,4920230,Afterworld,True,To be announced,0,0,0
1,1579420,Guardians of the Wild Sky,True,Coming soon,0,0,0
2,1573550,LEGO® Skylines,True,Coming soon,0,0,0
3,1257610,Prison Architect 2,True,To be announced,0,0,0
4,794820,Knights of Pen and Paper 2: Free Edition,False,"21 Feb, 2018",0,0,0


## 5. Recent reviews next to all-time reviews

- The all-time numbers come from `review_summary` of the same run, so both count all languages.
- The recent percent is NULL (not 0) when a game has no recent reviews.

**💡Insights:**

| Game | Reviews last 30 days | Positive last 30 days | Positive all time |
|---|---:|---:|---:|
| Transport Fever 3 | 3,766 | 70.8 % | 71.0 % |
| Hearts of Iron IV | 3,259 | 87.2 % | 90.5 % |
| Cities: Skylines | 1,476 | 90.1 % | 92.9 % |
| Cities: Skylines II | 730 | 70.7 % | 57.4 % |
| Crusader Kings III | 690 | 70.4 % | 90.4 % |
| Europa Universalis IV | 665 | 33.8 % | 86.5 % |
| Europa Universalis V | 653 | 38.7 % | 64.4 % |
| Stellaris | 559 | 65.1 % | 85.7 % |
| Victoria 3 | 328 | 59.5 % | 71.7 % |
| Prison Architect | 305 | 38.0 % | 89.0 % |
| Age of Wonders 4 | 172 | 80.2 % | 82.0 % |

- Only these 11 games have 100 or more reviews in the last 30 days.
- *Transport Fever 3* has 3,766 of its 7,202 reviews in the last 30 days (about 52 %, my calculation). I did not check why.
- *Stellaris* is 559 here. Earlier the same day the probe gave 558 (363 + 195) and the test run 559 (364 + 195). The number moves a little between requests, and I don't know why.
- This is one snapshot. It shows how things are now, not a trend.

In [9]:
query(f"""
SELECT g.name,
       s.total_reviews AS all_time_reviews,
       ROUND(100 * s.total_positive / NULLIF(s.total_reviews, 0), 1) AS pct_positive_all_time,
       r.recent_positive + r.recent_negative AS reviews_last_30d,
       ROUND(100 * r.recent_positive / NULLIF(r.recent_positive + r.recent_negative, 0), 1) AS pct_positive_last_30d
FROM review_recent r
JOIN review_summary s ON s.steam_appid = r.steam_appid AND s._dlt_load_id = r._dlt_load_id
JOIN games g ON g.steam_appid = r.steam_appid AND g._dlt_load_id = r._dlt_load_id
WHERE r._dlt_load_id = '{LOAD_ID}'
ORDER BY reviews_last_30d DESC
""")

,NAME,ALL_TIME_REVIEWS,PCT_POSITIVE_ALL_TIME,REVIEWS_LAST_30D,PCT_POSITIVE_LAST_30D
0,Transport Fever 3,7202,71.0,3766,70.8
1,Hearts of Iron IV,260509,90.5,3259,87.2
2,Cities: Skylines,219092,92.9,1476,90.1
3,Cities: Skylines II,64847,57.4,730,70.7
4,Crusader Kings III,94053,90.4,690,70.4
5,Europa Universalis IV,92188,86.5,665,33.8
6,Europa Universalis V,25449,64.4,653,38.7
7,Stellaris,139723,85.7,559,65.1
8,Victoria 3,41039,71.7,328,59.5
9,Prison Architect,59889,89.0,305,38.0


## 6. How many recent reviews do games have?

A percentage from 2 reviews says nothing. So this counts the games by how many reviews they got in the last 30 days.

**💡Insights:**

| Reviews in the last 30 days | Games |
|---|---:|
| 0 | 5 |
| 1-9 | 36 |
| 10-99 | 29 |
| 100 or more | 11 |

- 41 of the 81 games have fewer than 10 recent reviews. Only 11 have 100 or more.
- For most games the recent percent is noise, because one vote changes it a lot.
- We need a minimum number of recent reviews before we show it. Not decided yet.

In [10]:
query(f"""
SELECT
    CASE
        WHEN recent_total = 0 THEN '0 (none)'
        WHEN recent_total < 10 THEN '1 (1-9)'
        WHEN recent_total < 100 THEN '2 (10-99)'
        ELSE '3 (100+)'
    END AS recent_volume,
    COUNT(*) AS games
FROM (
    SELECT recent_positive + recent_negative AS recent_total
    FROM review_recent
    WHERE _dlt_load_id = '{LOAD_ID}'
)
GROUP BY recent_volume
ORDER BY recent_volume
""")

,RECENT_VOLUME,GAMES
0,0 (none),5
1,1 (1-9),36
2,2 (10-99),29
3,3 (100+),11


## 7. Check: recent can never be more than all-time

The last 30 days are part of all time. If a game had more recent reviews than all-time reviews, the two numbers would not count the same thing (for example, different languages).

**💡Insights:**
- The result is empty. No game has more recent than all-time reviews, so the two counts look consistent.

In [11]:
query(f"""
SELECT g.name, s.total_reviews AS all_time_reviews,
       r.recent_positive + r.recent_negative AS reviews_last_30d
FROM review_recent r
JOIN review_summary s ON s.steam_appid = r.steam_appid AND s._dlt_load_id = r._dlt_load_id
JOIN games g ON g.steam_appid = r.steam_appid AND g._dlt_load_id = r._dlt_load_id
WHERE r._dlt_load_id = '{LOAD_ID}'
  AND r.recent_positive + r.recent_negative > s.total_reviews
""")

,NAME,ALL_TIME_REVIEWS,REVIEWS_LAST_30D


## 8. Who is doing worse (or better) than usual?

- Change in points = percent positive in the last 30 days minus percent positive of all time.
- Only games with 100 or more recent reviews, because smaller numbers jump around.

**💡Insights:**

| Game | Change (points) | Recent reviews |
|---|---:|---:|
| Europa Universalis IV | -52.6 | 665 |
| Prison Architect | -51.0 | 305 |
| Europa Universalis V | -25.7 | 653 |
| Stellaris | -20.6 | 559 |
| Crusader Kings III | -20.0 | 690 |
| Victoria 3 | -12.2 | 328 |
| Hearts of Iron IV | -3.4 | 3,259 |
| Cities: Skylines | -2.8 | 1,476 |
| Age of Wonders 4 | -1.7 | 172 |
| Transport Fever 3 | -0.1 | 3,766 |
| Cities: Skylines II | +13.3 | 730 |

- 6 of the 11 games are more than 10 points below their all-time percent. Only Cities: Skylines II is clearly above.
- *Europa Universalis IV* fell from 86.5 % to 33.8 %, and *Prison Architect* from 89.0 % to 38.0 %. With 300 or more reviews behind each, that is not just noise.
- I don't know why. It could be a patch, a DLC or a review bomb. Nothing here checks it. News data could help later.

In [12]:
query(f"""
WITH compare AS (
    SELECT g.name,
           r.recent_positive + r.recent_negative AS reviews_last_30d,
           100 * s.total_positive / NULLIF(s.total_reviews, 0) AS pct_all_time,
           100 * r.recent_positive / NULLIF(r.recent_positive + r.recent_negative, 0) AS pct_last_30d
    FROM review_recent r
    JOIN review_summary s ON s.steam_appid = r.steam_appid AND s._dlt_load_id = r._dlt_load_id
    JOIN games g ON g.steam_appid = r.steam_appid AND g._dlt_load_id = r._dlt_load_id
    WHERE r._dlt_load_id = '{LOAD_ID}'
)
SELECT name, reviews_last_30d, ROUND(pct_all_time, 1) AS pct_all_time,
       ROUND(pct_last_30d, 1) AS pct_last_30d,
       ROUND(pct_last_30d - pct_all_time, 1) AS change_points
FROM compare
WHERE reviews_last_30d >= 100
ORDER BY change_points
""")

,NAME,REVIEWS_LAST_30D,PCT_ALL_TIME,PCT_LAST_30D,CHANGE_POINTS
0,Europa Universalis IV,665,86.5,33.8,-52.6
1,Prison Architect,305,89.0,38.0,-51.0
2,Europa Universalis V,653,64.4,38.7,-25.7
3,Stellaris,559,85.7,65.1,-20.6
4,Crusader Kings III,690,90.4,70.4,-20.0
5,Victoria 3,328,71.7,59.5,-12.2
6,Hearts of Iron IV,3259,90.5,87.2,-3.4
7,Cities: Skylines,1476,92.9,90.1,-2.8
8,Age of Wonders 4,172,82.0,80.2,-1.7
9,Transport Fever 3,3766,71.0,70.8,-0.1


## 9. The language fix in `review_summary`

- Until 2026-11-02 my code sent `language: "all"`. Steam ignores that, so `review_summary` counted **English reviews only**.
- Now it sends `languages: ["all"]`. The run on 2026-11-02 is the first with all languages.
- `review_recent` counts all languages from the start. Without the fix I would have compared an English-only total with an all-languages recent count.

**💡Insights:**

| Load | Sum of `total_reviews`, 81 games | Stellaris |
|---|---:|---:|
| 2026-10-26 | 674,088 | 76,804 |
| 2026-10-27 | 674,599 | 76,814 |
| 2026-10-30 | 674,829 | 76,828 |
| 2026-11-02 | 1,269,461 | 139,723 |

- In the first three loads (English only) the sum grew by about 740 in four days. Then it nearly doubled in one step. That jump is the fix, not new reviews.
- The old rows stay English only. I did not change them.
- So the series of totals has a break at 2026-11-02. dbt has to mark it (finding 6).

In [13]:
query("""
SELECT _dlt_load_id, MIN(collected_at) AS collected_at, COUNT(*) AS games,
       SUM(total_reviews) AS sum_of_total_reviews
FROM review_summary
GROUP BY _dlt_load_id
ORDER BY collected_at
""")

,_DLT_LOAD_ID,COLLECTED_AT,GAMES,SUM_OF_TOTAL_REVIEWS
0,1793014149.3994608,2026-10-26 11:29:09.930573+00:00,81,674088
1,1793094197.9862032,2026-10-27 09:43:18.514452+00:00,81,674599
2,1793369111.4503539,2026-10-30 14:05:12.029535+00:00,81,674829
3,1793621677.9089065,2026-11-02 12:14:38.451088+00:00,81,1269461


In [14]:
# One game over the runs: Stellaris.
query("""
SELECT _dlt_load_id, collected_at, total_reviews, total_positive, total_negative
FROM review_summary
WHERE steam_appid = 281990
ORDER BY collected_at
""")

,_DLT_LOAD_ID,COLLECTED_AT,TOTAL_REVIEWS,TOTAL_POSITIVE,TOTAL_NEGATIVE
0,1793014149.3994608,2026-10-26 11:29:09.930573+00:00,76804,66163,10641
1,1793094197.9862032,2026-10-27 09:43:18.514452+00:00,76814,66169,10645
2,1793369111.4503539,2026-10-30 14:05:12.029535+00:00,76828,66175,10653
3,1793621677.9089065,2026-11-02 12:14:38.451088+00:00,139723,119758,19965


## 10. Findings for dbt staging

- These are proposals for when we build the staging model. The SQL is a sketch, not tested. The model and column names are only suggestions.

---

### Finding 1: one row per game per load (staging)

- 81 rows, 81 games, no duplicates (sections 1 and 2).
- Keep `steam_appid`, `collected_at`, `day_range`, `recent_positive` and `recent_negative`. Rename `_dlt_load_id` to `load_id`.
- Tests: `unique_combination_of_columns` on (`steam_appid`, `load_id`), `not_null` on the columns, and `relationships` to the games model.
- If we ever load other windows (7 or 90 days), `day_range` must be part of the unique combination.

### Finding 2: dbt makes the total and the percent (staging)

- The table only has positive and negative.
- Add `recent_total`, a flag `has_recent_reviews`, and `pct_positive_recent`, which is NULL when the total is 0.

```sql
recent_positive + recent_negative as recent_total,
round(100 * recent_positive / nullif(recent_positive + recent_negative, 0), 1) as pct_positive_recent
```

### Finding 3: 0 is a number, not missing (staging)

- 5 games have 0 recent reviews, and the same 5 have 0 reviews of all time (section 4).
- Never turn a NULL percent into 0. A percent of 0 would say everybody voted down.
- Do not use "0 reviews" to say a game is unreleased. Use `coming_soon` from `games`.

### Finding 4: small numbers are noise (marts)

- 36 games have 1 to 9 recent reviews, 29 have 10 to 99, and only 11 have 100 or more (section 6).
- Do not filter in staging. In the marts, show the recent percent only above a minimum number of reviews.
- The minimum is not decided. With 100 only 11 games stay, with 10 about 40.

### Finding 5: the days overlap between runs (intermediate and marts)

- Every run counts the last 30 days, and the next run shares 29 of them.
- Never add `recent_total` over several days. It counts the same reviews many times.
- For "now", use the latest row per game. For a trend, draw one value per day.

### Finding 6: a break in `review_summary` at 2026-11-02 (staging)

- Older loads count English reviews only. This load counts all languages (section 9).
- In `stg_steam__review_summary`, add a flag like `counts_all_languages` (true from this load).
- When we compare with `review_recent`, use only the flagged rows. Write it in the thesis too.

### Finding 7: a test that recent is never above all-time (intermediate)

- Section 7 returned nothing.
- Add a test with `severity: warn` that returns the rows where `recent_total > total_reviews` for the same load. It passes when it returns nothing.

---

**Still open**

| Question | Status |
|---|---|
| Minimum number of recent reviews for a percent | Not decided |
| How much the review-bomb filter changes the counts | Not tested |
| Why Europa Universalis IV and Prison Architect fell so far | Unknown, needs news or patch dates |
| Why the count moves by 1 between requests (558, 559) | Unknown |
| Trend over days | Needs more runs |
| Rule for repeated runs (one per day?) | Proposal, not decided |

In [15]:
# conn.close()